# 06 · Close the loop: back to the MCAP timeline

**Agenda: 95–105 min.** Every frame knows its `episode_id` and `timestamp_ns`. So detector output on frames can be turned back into **temporal tags** on the episode — intervals where the gripper, or a brick, is visible — and queried exactly like the grasp/release tags we started with.

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F
import fiftyone.utils.huggingface as fouh

def load(name):
    # Local copy if download_data.py already ran, otherwise pull it from Hugging Face now
    if fo.dataset_exists(name):
        return fo.load_dataset(name)
    print(f"{name} not found locally; downloading from Hugging Face (one time)…")
    return fouh.load_from_hub(f"dgural/{name}", name=name, persistent=True)

episodes = load("droid-mcap-workshop")
frames = load("droid-frames-workshop")
print(episodes.temporal_tags.count())

The dataset already carries `brick-visible` and `gripper-visible` tags derived from the Nebius-trained YOLO11n. Here is how they were built, and you can rebuild them for any label:

In [ ]:
from fiftyone.core.tags import TemporalTag

def intervals_from_predictions(frames, episodes, label, field="yolo11n_preds", min_conf=0.5, camera="ext1", gap_s=1.5):
    """Detections on 1-fps frames -> merged [start, end] intervals per episode (ns, episode-relative)."""
    tags = []
    ep_ids = {s.episode_id: s.id for s in episodes.select_fields("episode_id")}
    hits = (frames.match(F("camera") == camera)
                  .filter_labels(field, (F("label") == label) & (F("confidence") >= min_conf), only_matches=True)
                  .match(F("episode_id").is_in(list(ep_ids))))
    by_ep = {}
    for ep, t in zip(*hits.values(["episode_id", "t_rel_s"])):
        by_ep.setdefault(ep, []).append(t)
    for ep, ts in by_ep.items():
        ts.sort()
        start = prev = ts[0]
        for t in ts[1:] + [None]:
            if t is None or t - prev > gap_s:
                tags.append(TemporalTag(sample_id=ep_ids[ep], start=int(start * 1e9), end=int((prev + 1.0) * 1e9),
                                        tag=f"{label}-visible", index_type=2))
                if t is not None:
                    start = t
            if t is not None:
                prev = t
    return tags

new_tags = intervals_from_predictions(frames, episodes, "brick")
print(len(new_tags), "brick-visible intervals across", len({t.sample_id for t in new_tags}), "episodes")

In [ ]:
# Write them (idempotent: clear the tag first)
episodes.temporal_tags.delete(tags="brick-visible-mine") if "brick-visible-mine" in episodes.temporal_tags.count() else None
for t in new_tags:
    t.tag = "brick-visible-mine"
episodes.temporal_tags.add(new_tags)
print(episodes.temporal_tags.count())

In [ ]:
session = fo.launch_app(episodes)
session.view = episodes.match_temporal_tags(tags="brick-visible")

Open an episode: the timeline now has `grasp`, `release`, and `brick-visible` rows. Scrub to a `brick-visible` interval — the external camera should show the brick.

## Rank episodes by what the detector saw

In [ ]:
# Episodes where the model is least confident about the gripper: candidates for more data or review
conf = (frames.match(F("camera") == "ext1")
              .filter_labels("yolo11n_preds", F("label") == "gripper", only_matches=True)
              .match(F("episode_id").is_in(episodes.distinct("episode_id"))))
per_ep = {}
for ep, dets in zip(*conf.values(["episode_id", "yolo11n_preds.detections.confidence"])):
    per_ep.setdefault(ep, []).extend(dets)
for ep, cs in sorted(per_ep.items(), key=lambda kv: sum(kv[1]) / len(kv[1])):
    print(f"{sum(cs)/len(cs):.2f}  {len(cs):3d} gripper boxes  {ep}")

### The loop

**see** the recording → **curate** frames with robot state → **embed** and find the odd ones → **auto-label** → **train** on Nebius → **evaluate** → **tag the recording** with what the model found → the next collection run is curated with these saved views and tags.

Same tools, same loop, at 100 episodes on a laptop or millions in a deployment.

- Try it on your own recordings: drop any `.mcap` into the **MCAP Explorer** panel.
- Hosted FiftyOne: [app.voxel51.com](https://app.voxel51.com)
- GPUs for the loop: [Nebius Builder Program](https://link.voxel51.com/nebius)